# ShotGuide CLIP Embedding Baseline

This notebook extracts frozen CLIP ViT-B/32 image embeddings and trains a small multi-task head on top of them.

- Uses the existing video-level split from `outputs_baseline/dataset_index_with_splits.csv`
- Extracts one 512-dim CLIP embedding per labeled image
- Saves embeddings for reuse
- Trains heads for `shot_type` and `has_text`


In [ ]:
from pathlib import Path
from collections import Counter
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset, TensorDataset
from tqdm.auto import tqdm
import open_clip

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

ROOT = Path.cwd()
SPLIT_CSV = ROOT / 'outputs_baseline' / 'dataset_index_with_splits.csv'
OUTPUT_DIR = ROOT / 'outputs_clip_embeddings'
OUTPUT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

## 1. Load the Fixed Split

The split is reused from the previous ResNet baseline so the comparison remains fair.

In [ ]:
df = pd.read_csv(SPLIT_CSV)
df['has_text'] = df['has_text'].astype(int)

valid_splits = {'train', 'val', 'test'}
assert set(df['split'].unique()) == valid_splits, df['split'].unique()

leaks = [video_id for video_id, g in df.groupby('video_id') if g['split'].nunique() > 1]
print('rows:', len(df))
print('videos:', df['video_id'].nunique())
print('group leakage count:', len(leaks))
assert len(leaks) == 0

display(pd.crosstab(df['joint_label'], df['split']))
display(df.head())

## 2. Load Frozen CLIP ViT-B/32

The first run may download pretrained OpenAI CLIP weights.

In [ ]:
MODEL_NAME = 'ViT-B-32'
PRETRAINED = 'openai'

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=DEVICE)
model.eval()

for param in model.parameters():
    param.requires_grad = False

print('model:', MODEL_NAME, PRETRAINED)
print('embedding dim:', model.visual.output_dim)

## 3. Extract and Save CLIP Embeddings

In [ ]:
class ImagePathDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        path = self.frame.loc[idx, 'filepath']
        image = Image.open(path).convert('RGB')
        return self.transform(image), idx

EMBEDDING_NPZ = OUTPUT_DIR / 'clip_vit_b32_openai_embeddings.npz'
EMBEDDING_META_CSV = OUTPUT_DIR / 'clip_embedding_metadata.csv'

if EMBEDDING_NPZ.exists() and EMBEDDING_META_CSV.exists():
    print('Loading cached embeddings:', EMBEDDING_NPZ)
    cached = np.load(EMBEDDING_NPZ)
    embeddings = cached['embeddings']
else:
    loader = DataLoader(ImagePathDataset(df, preprocess), batch_size=64, shuffle=False, num_workers=0)
    embedding_chunks = []
    order = []

    with torch.no_grad():
        for images, indices in tqdm(loader):
            images = images.to(DEVICE)
            features = model.encode_image(images)
            features = features / features.norm(dim=-1, keepdim=True)
            embedding_chunks.append(features.cpu().float().numpy())
            order.extend(indices.numpy().tolist())

    embeddings = np.concatenate(embedding_chunks, axis=0).astype('float32')
    order = np.array(order)
    assert np.all(order == np.arange(len(df)))

    np.savez_compressed(
        EMBEDDING_NPZ,
        embeddings=embeddings,
        has_text=df['has_text'].to_numpy(dtype=np.int64),
        split=df['split'].to_numpy(),
        shot_type=df['shot_type'].to_numpy(),
        filepath=df['filepath'].to_numpy(),
    )
    df.assign(embedding_index=np.arange(len(df))).to_csv(EMBEDDING_META_CSV, index=False, encoding='utf-8-sig')

print('embedding shape:', embeddings.shape)
print('saved:', EMBEDDING_NPZ)

## 4. Train a Multi-task Head on Frozen Embeddings

In [ ]:
SHOT_TYPES = ['close-up', 'medium', 'object', 'space', 'wide']
shot_to_idx = {name: i for i, name in enumerate(SHOT_TYPES)}
idx_to_shot = {i: name for name, i in shot_to_idx.items()}

shot_labels = df['shot_type'].map(shot_to_idx).to_numpy(dtype=np.int64)
text_labels = df['has_text'].to_numpy(dtype=np.int64)

def split_arrays(split_name):
    mask = df['split'].to_numpy() == split_name
    x = torch.tensor(embeddings[mask], dtype=torch.float32)
    y_shot = torch.tensor(shot_labels[mask], dtype=torch.long)
    y_text = torch.tensor(text_labels[mask], dtype=torch.long)
    return x, y_shot, y_text, mask

x_train, y_shot_train, y_text_train, train_mask = split_arrays('train')
x_val, y_shot_val, y_text_val, val_mask = split_arrays('val')
x_test, y_shot_test, y_text_test, test_mask = split_arrays('test')

train_loader = DataLoader(TensorDataset(x_train, y_shot_train, y_text_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_shot_val, y_text_val), batch_size=256, shuffle=False)
test_loader = DataLoader(TensorDataset(x_test, y_shot_test, y_text_test), batch_size=256, shuffle=False)

x_train.shape, x_val.shape, x_test.shape

In [ ]:
class ClipEmbeddingMultiTaskHead(nn.Module):
    def __init__(self, embedding_dim=512, hidden_dim=256, num_shot_classes=5, dropout=0.20):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.shot_head = nn.Linear(hidden_dim, num_shot_classes)
        self.text_head = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        z = self.shared(x)
        return self.shot_head(z), self.text_head(z)

def make_class_weights(labels, num_classes):
    counts = Counter(labels.tolist())
    total = sum(counts.values())
    return torch.tensor([total / (num_classes * max(counts.get(i, 0), 1)) for i in range(num_classes)], dtype=torch.float32)

head = ClipEmbeddingMultiTaskHead(embedding_dim=embeddings.shape[1], hidden_dim=256).to(DEVICE)
shot_criterion = nn.CrossEntropyLoss(weight=make_class_weights(y_shot_train, len(SHOT_TYPES)).to(DEVICE))
text_criterion = nn.CrossEntropyLoss(weight=make_class_weights(y_text_train, 2).to(DEVICE))
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-3)

TEXT_LOSS_WEIGHT = 1.0
EPOCHS = 80
PATIENCE = 12

def evaluate(loader):
    head.eval()
    total_loss = 0.0
    shot_true, shot_pred = [], []
    text_true, text_pred = [], []
    with torch.no_grad():
        for x, y_shot, y_text in loader:
            x = x.to(DEVICE)
            y_shot = y_shot.to(DEVICE)
            y_text = y_text.to(DEVICE)
            shot_logits, text_logits = head(x)
            loss = shot_criterion(shot_logits, y_shot) + TEXT_LOSS_WEIGHT * text_criterion(text_logits, y_text)
            total_loss += loss.item() * x.size(0)
            shot_true.extend(y_shot.cpu().numpy().tolist())
            shot_pred.extend(shot_logits.argmax(1).cpu().numpy().tolist())
            text_true.extend(y_text.cpu().numpy().tolist())
            text_pred.extend(text_logits.argmax(1).cpu().numpy().tolist())
    return {
        'loss': total_loss / len(loader.dataset),
        'shot_acc': accuracy_score(shot_true, shot_pred),
        'shot_macro_f1': f1_score(shot_true, shot_pred, average='macro', zero_division=0),
        'text_acc': accuracy_score(text_true, text_pred),
        'text_f1': f1_score(text_true, text_pred, average='binary', zero_division=0),
        'joint_acc': np.mean((np.array(shot_true) == np.array(shot_pred)) & (np.array(text_true) == np.array(text_pred))),
    }

def train_one_epoch(loader):
    head.train()
    for x, y_shot, y_text in loader:
        x = x.to(DEVICE)
        y_shot = y_shot.to(DEVICE)
        y_text = y_text.to(DEVICE)
        shot_logits, text_logits = head(x)
        loss = shot_criterion(shot_logits, y_shot) + TEXT_LOSS_WEIGHT * text_criterion(text_logits, y_text)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

history = []
best_state = None
best_val_joint = -1.0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_one_epoch(train_loader)
    train_metrics = evaluate(train_loader)
    val_metrics = evaluate(val_loader)
    row = {'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_{k}': v for k, v in val_metrics.items()}}
    history.append(row)
    if epoch == 1 or epoch % 5 == 0:
        print(row)

    if val_metrics['joint_acc'] > best_val_joint:
        best_val_joint = val_metrics['joint_acc']
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('Early stopping at epoch', epoch)
            break

head.load_state_dict(best_state)
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / 'clip_head_training_history.csv', index=False)

checkpoint_path = OUTPUT_DIR / 'clip_vit_b32_multitask_head.pt'
torch.save({
    'model_state_dict': best_state,
    'shot_to_idx': shot_to_idx,
    'idx_to_shot': idx_to_shot,
    'clip_model': MODEL_NAME,
    'clip_pretrained': PRETRAINED,
}, checkpoint_path)

print('best val joint acc:', best_val_joint)
print('saved:', checkpoint_path)
history_df.tail()

## 5. Test Evaluation

In [ ]:
head.eval()
shot_true, shot_pred = [], []
text_true, text_pred = [], []
shot_conf, text_prob = [], []

with torch.no_grad():
    for x, y_shot, y_text in test_loader:
        x = x.to(DEVICE)
        shot_logits, text_logits = head(x)
        shot_p = torch.softmax(shot_logits, dim=1)
        text_p = torch.softmax(text_logits, dim=1)
        shot_true.extend(y_shot.numpy().tolist())
        text_true.extend(y_text.numpy().tolist())
        shot_pred.extend(shot_p.argmax(1).cpu().numpy().tolist())
        text_pred.extend(text_p.argmax(1).cpu().numpy().tolist())
        shot_conf.extend(shot_p.max(1).values.cpu().numpy().tolist())
        text_prob.extend(text_p[:, 1].cpu().numpy().tolist())

test_metrics = {
    'shot_acc': accuracy_score(shot_true, shot_pred),
    'shot_macro_f1': f1_score(shot_true, shot_pred, average='macro', zero_division=0),
    'text_acc': accuracy_score(text_true, text_pred),
    'text_f1': f1_score(text_true, text_pred, average='binary', zero_division=0),
    'joint_acc': np.mean((np.array(shot_true) == np.array(shot_pred)) & (np.array(text_true) == np.array(text_pred))),
}

with open(OUTPUT_DIR / 'clip_test_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(test_metrics, f, indent=2)

print(json.dumps(test_metrics, indent=2))
print('\nShot type report')
print(classification_report(shot_true, shot_pred, target_names=SHOT_TYPES, zero_division=0))
print('\nText report')
print(classification_report(text_true, text_pred, target_names=['notext', 'text'], zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

shot_cm = confusion_matrix(shot_true, shot_pred, labels=list(range(len(SHOT_TYPES))))
sns.heatmap(shot_cm, annot=True, fmt='d', cmap='Blues', xticklabels=SHOT_TYPES, yticklabels=SHOT_TYPES, ax=axes[0])
axes[0].set_title('CLIP Shot Type Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

text_cm = confusion_matrix(text_true, text_pred, labels=[0, 1])
sns.heatmap(text_cm, annot=True, fmt='d', cmap='Greens', xticklabels=['notext', 'text'], yticklabels=['notext', 'text'], ax=axes[1])
axes[1].set_title('CLIP Text Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'clip_confusion_matrices.png', dpi=160)
plt.show()

In [ ]:
test_result_df = df[test_mask].copy().reset_index(drop=True)
test_result_df['pred_shot_type'] = [idx_to_shot[i] for i in shot_pred]
test_result_df['pred_has_text'] = text_pred
test_result_df['shot_confidence'] = shot_conf
test_result_df['text_probability'] = text_prob
test_result_df['shot_correct'] = test_result_df['shot_type'] == test_result_df['pred_shot_type']
test_result_df['text_correct'] = test_result_df['has_text'].astype(int) == test_result_df['pred_has_text']
test_result_df['joint_correct'] = test_result_df['shot_correct'] & test_result_df['text_correct']
test_result_df.to_csv(OUTPUT_DIR / 'clip_test_predictions.csv', index=False, encoding='utf-8-sig')

display(test_result_df[['filename', 'video_id', 'shot_type', 'pred_shot_type', 'has_text', 'pred_has_text', 'shot_confidence', 'text_probability', 'joint_correct']].head(20))

## 6. Compare Against the ResNet Baseline

In [ ]:
resnet_metrics = {
    'shot_acc': 0.7104247104247104,
    'shot_macro_f1': 0.7071751112269045,
    'text_acc': 0.6602316602316602,
    'text_f1': 0.7621621621621621,
    'joint_acc': 0.4980694980694981,
}

comparison = pd.DataFrame([
    {'model': 'ResNet18 frozen backbone baseline', **resnet_metrics},
    {'model': 'CLIP ViT-B/32 frozen embeddings', **test_metrics},
])
comparison.to_csv(OUTPUT_DIR / 'resnet_vs_clip_comparison.csv', index=False)
comparison